# Colab smoke test\n\nGitHubから開いて実行できる最小テストです。外部APIキーやプライベートデータは不要です。

In [ ]:
!pip -q install pandas pyarrow scikit-learn\nprint('Dependencies installed')

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\nimport zipfile\nROOT=Path('/content/sports_prediction_smoke')\n(ROOT/'raw').mkdir(parents=True,exist_ok=True)\n(ROOT/'features').mkdir(parents=True,exist_ok=True)\ngames=pd.DataFrame({\n    'match_id':['demo-001','demo-002','demo-003','demo-004'],\n    'datetime':pd.to_datetime(['2024-01-01','2024-01-03','2024-01-05','2024-01-07'],utc=True),\n    'home_team':['A','B','A','C'],\n    'away_team':['B','A','C','A'],\n    'home_score':[2,1,3,0],\n    'away_score':[1,1,0,2],\n    'source':['smoke_test']*4,\n})\ngames.to_parquet(ROOT/'raw'/'games.parquet',index=False)\ngames=games.sort_values(['datetime','match_id']).copy()\ngames['home_win_prior']=games.groupby('home_team')['home_score'].transform(lambda s:s.shift().expanding().mean())\ngames['away_win_prior']=games.groupby('away_team')['away_score'].transform(lambda s:s.shift().expanding().mean())\ngames['prediction_cutoff_at']=games['datetime']-pd.Timedelta(hours=1)\ngames.to_parquet(ROOT/'features'/'pre_match_features.parquet',index=False)\nassert games['datetime'].is_monotonic_increasing\nassert (pd.to_datetime(games['prediction_cutoff_at'],utc=True)<games['datetime']).all()\nprint('Rows:',len(games))\nprint('Chronology and cutoff checks: PASS')

In [ ]:
archive='/content/sports_prediction_smoke.zip'\nwith zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:\n    for f in ROOT.rglob('*'):\n        if f.is_file(): z.write(f,f.relative_to(ROOT))\nprint('Created',archive)\nfrom google.colab import files\nfiles.download(archive)